# FOXF1_bead — 00_manifest_qc

**Feeds:** Fig 2c, ED Fig 3d

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 00 | Manifest And Metadata QC

## Cell Guide
1. Resolve the workspace root and import the manifest builder.
2. Rebuild filesystem inventory, CZI metadata, and analysis manifests from raw data.
3. Inspect cohort counts and spot-check the regenerated selection tables.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Resolve project root whether notebook is launched from repo root or /notebooks.
CWD = Path.cwd().resolve()
if (CWD / "scripts").exists() and (CWD / "results").exists():
    ROOT = CWD
elif (CWD.parent / "scripts").exists() and (CWD.parent / "results").exists():
    ROOT = CWD.parent
else:
    ROOT = CWD

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import build_analysis_manifests as bam

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

print("ROOT:", ROOT)
print("Python:", sys.executable)


## Settings Explained

- This notebook rebuilds the manifest layer from the raw `data/` tree.
- The 2026-01-22 live include/exclude rule is taken from
  `data/2026-01-22_PDMS/day2/analysis_manifest_day2_single.tsv`.
- The selection rule remains `small_primary_large_spatial_when_available`.


In [ ]:
# Parameters
MANIFEST_OUTDIR = ROOT / "results" / "manifests"
WRITE_OUTPUTS = True
VERBOSE = True

print("MANIFEST_OUTDIR:", MANIFEST_OUTDIR)
print("WRITE_OUTPUTS:", WRITE_OUTPUTS)
print("VERBOSE:", VERBOSE)


In [ ]:
# Rebuild manifest outputs from raw data.
manifest_res = bam.run_manifest_pipeline(
    root=ROOT,
    outdir=MANIFEST_OUTDIR,
    write_outputs=WRITE_OUTPUTS,
    verbose=VERBOSE,
)

filesystem_df = manifest_res["filesystem_df"]
czi_metadata_df = manifest_res["czi_metadata_df"]
file_manifest_df = manifest_res["file_manifest_df"]
position_manifest_df = manifest_res["position_manifest_df"]
crosswalk_df = manifest_res["crosswalk_df"]

print(manifest_res["summary_text"])


In [ ]:
# Cohort-level manifest QC.
cohort_summary = position_manifest_df.groupby(["cohort_id", "pair_status"], as_index=False).size()
print("Position rows:", len(position_manifest_df))
print("File rows:", len(file_manifest_df))
display(cohort_summary)

primary_counts = file_manifest_df.groupby("cohort_id", as_index=False).agg(
    files_total=("file_path", "size"),
    files_primary=("include_primary_analysis", "sum"),
    files_spatial=("include_spatial_metadata", "sum"),
)
display(primary_counts)


In [ ]:
# Spot-check the 3-3 special case and the live/fix crosswalk.
special_33 = file_manifest_df[file_manifest_df["canonical_position"] == "3-3"].copy()
special_33 = special_33[[
    "cohort_id",
    "variant",
    "file_path",
    "include_primary_analysis",
    "include_spatial_metadata",
    "external_manifest_included",
    "external_manifest_note",
]]
display(special_33)

display(crosswalk_df.head(12))


In [ ]:
# Output paths written by this notebook.
display(pd.Series(manifest_res["output_paths"], name="path"))
